In [1]:
import os
import random
from pathlib import Path
from typing import Union

import h5py
import numpy as np
import torch
from datasets import *
from data_utils import *
from fastmri.data.subsample import EquiSpacedMaskFunc, RandomMaskFunc
from fastmri.data.transforms import tensor_to_complex_np, to_tensor
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import config
from torch.utils.data import DataLoader, TensorDataset
from pisco import *

file_data = '/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_train/file_brain_AXT1POST_203_6000861.h5'
dataset = KCoordDataset(file_data, n_slices=3, n_volumes=1, with_mask=True, acceleration=3, center_frac=0.15, mask_type='Random')




Training with Random mask


In [3]:
Nm, Nc, Nn, Ns = 1230, 4, 8, 100  # Example dimensions
matrix1 = torch.rand(Nm, Nc)
matrix2 = torch.rand(Nm, Nc * Nn)

subset1, subset2 = split_matrices_randomly(matrix1, matrix2, Ns)
# for i, (batch1, batch2) in enumerate(batches):
#     print(f"Batch {i+1}: {batch1.shape}, {batch2.shape}")


In [6]:
num_matrices = 5
Ws = [torch.rand(Nn * Nc, Nc) + 1j * torch.rand(Nn * Nc, Nc) for _ in range(num_matrices)]

In [7]:
total_loss = 0
Ns = len(Ws)
for i in range(Ns):
    for j in range(i+1, Ns):
        print(f'mat{i} vs mat{j}')
        diff = Ws[i].flatten() - Ws[j].flatten()
        ldist = torch.linalg.norm(diff.real, ord =1) + torch.linalg.norm(diff.imag, ord =1)

        total_loss += ldist
        
pisco_loss = total_loss/Ns**2

mat0 vs mat1
mat0 vs mat2
mat0 vs mat3
mat0 vs mat4
mat1 vs mat2
mat1 vs mat3
mat1 vs mat4
mat2 vs mat3
mat2 vs mat4
mat3 vs mat4


In [2]:

shape = dataset.metadata[0]["shape"]
center_data = dataset.metadata[0]["center"]
left_idx, right_idx, center_vals = (
    center_data["left_idx"],
    center_data["right_idx"],
    center_data["vals"])

n_slices, n_coils, width, height = shape

# Create tensors of indices for each dimension
kx_ids = torch.cat([torch.arange(left_idx), torch.arange(right_idx, width)])
ky_ids = torch.arange(height)
kz_ids = torch.arange(n_slices)
coil_ids = torch.arange(n_coils)

# Use meshgrid to create expanded grids
kspace_ids = torch.meshgrid(kx_ids, ky_ids, kz_ids, coil_ids, indexing="ij")
kspace_ids = torch.stack(kspace_ids, dim=-1).reshape(-1, len(kspace_ids))

dataset_new = TensorDataset(kspace_ids)
dataloader = DataLoader(
    dataset_new, batch_size=60_000, shuffle=False, num_workers=3
)

In [6]:
file = dataset.metadata[0]["file"]
with h5py.File(file, "r") as hf:
    ground_truth = hf["reconstruction_rss"][()][
        :n_slices
    ]
    kspace_gt = to_tensor(preprocess_kspace(hf["kspace"][()][:n_slices]))


In [5]:
from pisco import *
grappa_volume = torch.zeros(shape, dtype = torch.complex64)
norm_cte = [width, height, n_slices, n_coils]
volume_kspace = torch.zeros(
            (n_slices, n_coils, height, width),
            dtype=torch.float32,
        )
# If it is a checkpoint, recalculate the grappa volume, as the mean of the list of grappa matrixes 
# w_grappa = torch.tensor(np.mean(batch_grappas, axis=0)) # Size: Nn·Nc x Nc
    
# Now predict the sensitivities (accuracy of the Ws grappa matrixes)
# kspace_gt = tensor_to_complex_np(kspace_gt)
for points_ids in dataloader:
    points_ids = points_ids[0]
    t_coors, nn_coors, Nn = get_grappa_matrixes(points_ids, shape, patch_size=9, normalized=False)
    
    den_t_coors = torch.zeros(t_coors.shape, dtype = torch.int)
    den_nn_coors = torch.zeros(nn_coors.shape, dtype = torch.int)
    
    for idx in range(len(shape)):
        den_t_coors[...,idx] = denormalize_fn(t_coors[...,idx], norm_cte[idx]).int()
        den_nn_coors[...,idx] = denormalize_fn(nn_coors[...,idx], norm_cte[idx]).int()
    

    nn_kspacevals = torch.tensor(tensor_to_complex_np(kspace_gt[den_nn_coors[...,2], den_nn_coors[...,3], den_nn_coors[...,1], den_nn_coors[...,0]]))
    # nn_kspacevals = nn_kspacevals.reshape((t_coors.shape[0], Nn, n_coils))
#     
    ps_kspacevals = nn_kspacevals.view(t_coors.shape[0], Nn*n_coils)
    # t_kspacevals = torch.matmul(ps_kspacevals, w_grappa)  # NOTE : Computed value based on neighbouring patch of 3x3 and estimated grappa mean
    
#     grappa_volume[den_t_coors[...,2], den_t_coors[...,3], den_t_coors[...,1], den_t_coors[...,0]] = t_kspacevals
    
# grappa_img =  rss(inverse_fft2_shift((grappa_volume)))   
            

NameError: name 'kspace_gt' is not defined

In [93]:
from fastmri.data.transforms import tensor_to_complex_np, to_tensor

vol_id = 0
file = file_data
n_volumes = 1
n_slices = 1
with_mask = False
acceleration = 3
center_frac = 0.15
mask_type = 'Random'


with h5py.File(file, "r") as hf:
    volume_kspace = to_tensor(preprocess_kspace(hf["kspace"][()]))[:n_slices]

##################################################
# Mask creation
##################################################
if mask_type == "Random":
    mask_func = RandomMaskFunc(
    center_fractions=[center_frac], accelerations=[acceleration]
)
elif mask_type == "Equispaced": 
    mask_func = EquiSpacedMaskFunc(
    center_fractions=[center_frac], accelerations=[acceleration])
    
shape = (1,) * len(volume_kspace.shape[:-3]) + tuple(
    volume_kspace.shape[-3:])
mask, _ = mask_func(
    shape, None, vol_id
)  # use the volume index as random seed.

# mask, left_idx, right_idx = remove_center(mask)
_, left_idx, right_idx = remove_center(mask)  # NOTE: Uncomment to include the center region in the training data. Note that 'left_idx' and 'right_idx' are still needed.

##################################################
# Computing the indices
##################################################
n_slices, n_coils, height, width = volume_kspace.shape[:-1]
if with_mask:
    kx_ids = torch.where(mask.squeeze())[0]
else:
    kx_ids = torch.arange(width)
    # kx_ids = torch.from_numpy(np.setdiff1d(np.arange(width), np.arange(left_idx, right_idx))) # NOTE: Uncomment to include all the datapoints (fully-sampled volume), with the exception of the center region.
ky_ids = torch.arange(height)
kz_ids = torch.arange(n_slices)
coil_ids = torch.arange(n_coils)

kspace_ids = torch.meshgrid(kx_ids, ky_ids, coil_ids, indexing="ij")
kspace_ids = torch.stack(kspace_ids, dim=-1).reshape(-1, len(kspace_ids))

##################################################
# Computing the inputs
##################################################
# Convert indices into normalized coordinates in [-1, 1].
kspace_coords = torch.zeros((kspace_ids.shape[0], 3), dtype=torch.float)
kspace_coords[:, 0] = (2 * kspace_ids[:, 0]) / (width - 1) - 1
kspace_coords[:, 1] = (2 * kspace_ids[:, 1]) / (height - 1) - 1
kspace_coords[:, 2] = (2 * kspace_ids[:, 2]) / (n_coils - 1) - 1
# kspace_coords[:, 3] = (2 * kspace_ids[:, 3]) / (n_coils - 1) - 1

In [3]:
dataloader = DataLoader(dataset, batch_size=120000, shuffle=True, pin_memory=False)
print(shape)
counter = 0
for inputs, _ in dataloader:
    counter += 1
    
    #### Compute grid 
    t_coordinates, patch_coordinates, Nn = get_grappa_matrixes(inputs, shape, patch_size=9, normalized=True)
    
    if counter > 0:
        break

(3, 4, 320, 320)


In [9]:
patch_coordinates.shape

torch.Size([118911, 8, 4, 4])

In [24]:
a = torch.zeros(10, 4*4)

torch.eye(a.shape[1])

tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0

In [17]:
for idx in coil_ids:
    for nn in range(Nn):
        print(patch_coordinates[0, nn, idx, :])
        


tensor([310.,  51.,   2.,   0.])
tensor([311.,  51.,   2.,   0.])
tensor([312.,  51.,   2.,   0.])
tensor([310.,  52.,   2.,   0.])
tensor([312.,  52.,   2.,   0.])
tensor([310.,  53.,   2.,   0.])
tensor([311.,  53.,   2.,   0.])
tensor([312.,  53.,   2.,   0.])
tensor([310.,  51.,   2.,   1.])
tensor([311.,  51.,   2.,   1.])
tensor([312.,  51.,   2.,   1.])
tensor([310.,  52.,   2.,   1.])
tensor([312.,  52.,   2.,   1.])
tensor([310.,  53.,   2.,   1.])
tensor([311.,  53.,   2.,   1.])
tensor([312.,  53.,   2.,   1.])
tensor([310.,  51.,   2.,   2.])
tensor([311.,  51.,   2.,   2.])
tensor([312.,  51.,   2.,   2.])
tensor([310.,  52.,   2.,   2.])
tensor([312.,  52.,   2.,   2.])
tensor([310.,  53.,   2.,   2.])
tensor([311.,  53.,   2.,   2.])
tensor([312.,  53.,   2.,   2.])
tensor([310.,  51.,   2.,   3.])
tensor([311.,  51.,   2.,   3.])
tensor([312.,  51.,   2.,   3.])
tensor([310.,  52.,   2.,   3.])
tensor([312.,  52.,   2.,   3.])
tensor([310.,  53.,   2.,   3.])
tensor([31

In [37]:
patch_coordinates.shape

torch.Size([118850, 8, 4, 4])

In [14]:
### Get the grid for computing PISCO 
dataloader = DataLoader(dataset, batch_size=120000, shuffle=True, pin_memory=False)
count = 0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_coils = 8
n_slices = 3
width = 320
height = 320

for inputs, targets in dataloader:
    inputs, targets = inputs.to(device), targets.to(device)
    count += 1
    if count > 1:
        break

k_coors = torch.zeros((inputs.shape[0], 4), dtype=torch.float)
k_coors[:,0] = denormalize(inputs[:,0], width)
k_coors[:,1] = denormalize(inputs[:,1], height)
k_coors[:,2] = denormalize(inputs[:,2], n_slices)
k_coors[:,3] = denormalize(inputs[:,3], n_coils)


# Remove the edges 
leftmost_vedge = (k_coors[:, 1] == 0)
rightmost_vedge = (k_coors[:, 1] == 319)
upmost_vedge = (k_coors[:, 0] == 0)
downmost_vedge = (k_coors[:, 0] == 319)

edges = leftmost_vedge | rightmost_vedge | upmost_vedge | downmost_vedge
k_nedge = k_coors[~edges]

# #### Reshape:
# # Reshape input matrixes for coilID to be considered dim : n_points x N_coils x 4
r_kcoors = np.repeat(k_nedge[:, np.newaxis, :], n_coils, axis=1)
r_kcoors[...,-1] = torch.arange(n_coils)

# ##### Reshape patches matrix to : n_points x n_neighbours x N_coils x 4
build_neighbours = get_patch()
patch_coors = build_neighbours(r_kcoors)

# Reshape so that dim : n_points x N_n x Nc x 4 (kx,ky,kz, n_coils coordinates)
r_patch = torch.zeros((patch_coors.shape[0],patch_coors.shape[1], r_kcoors.shape[2]))
r_patch[...,:3] = patch_coors
r_patch = np.repeat(r_patch[:, :, np.newaxis], n_coils, axis=2)
r_patch[...,-1] = torch.arange(n_coils)

### For predicting, normalize coordinates back to [-1,1]
# Normalize the NP neighbourhood coordinates
n_r_patch = torch.zeros((r_patch.shape), dtype=torch.float)
n_r_patch[:,:,:,0] = normalize(r_patch[:,:,:,0], width)
n_r_patch[:,:,:,1] = normalize(r_patch[:,:,:,1], height)
n_r_patch[:,:,:,2] = normalize(r_patch[:,:,:,2], n_slices)
n_r_patch[:,:,:,3] = normalize(r_patch[:,:,:,3], n_coils)
# Flatten the first dimensions for the purpose of kvalue prediction
Nn = n_r_patch.shape[1]
n_r_patch = n_r_patch.view(-1, n_coils, 4)

# Normalize the Nt targets coordinates
n_r_koors = torch.zeros((r_kcoors.shape), dtype=torch.float)
n_r_koors[:,:,0] = normalize(r_kcoors[:,:,0], width)
n_r_koors[:,:,1] = normalize(r_kcoors[:,:,1], height)
n_r_koors[:,:,2] = normalize(r_kcoors[:,:,2], n_slices)
n_r_koors[:,:,3] = normalize(r_kcoors[:,:,3], n_coils)

In [31]:
from model import *
model = Siren()
size_minibatch = 1000

t_predicted = torch.zeros((n_r_koors.shape[0], n_coils), dtype=torch.complex64)
patch_predicted = torch.zeros((n_r_patch.shape[0], n_coils), dtype=torch.complex64)

# for coil_id in range(n_coils):
    # t_predicted[:,coil_id] = torch.view_as_complex(model(n_r_koors[:,coil_id,:]))
    # patch_predicted[:,coil_id] = torch.view_as_complex(model(n_r_patch[:,coil_id,:]))

# # Reshape back the patches_matrix
patch_predicted = patch_predicted.view(n_r_koors.shape[0], Nn, n_coils)

# size_minibatch = 300
T_s, Ns = split_batch(t_predicted, size_minibatch)
P_s, Ns = split_batch(patch_predicted, size_minibatch)


# ######## Here compute the Lpisco

In [32]:
P_s[0].shape

torch.Size([1000, 8, 8])

In [42]:
## L pisco
##################################
alpha = 1.e-4
Ws = []

# Generate the list of Ws from the subset of minibatches 
for i, t_s in enumerate(T_s):
    p_s = P_s[i]
    p_s = torch.flatten(p_s, start_dim=1)
    print()
    ws = compute_Lsquares(p_s, t_s, alpha)
    print(ws.shape)
    Ws.append(ws)


pisco_loss = L_pisco (Ws) # Ws is a list of Ws' from the minibatches

print(pisco_loss)


torch.Size([32, 4])

torch.Size([32, 4])

torch.Size([32, 4])

torch.Size([32, 4])

torch.Size([32, 4])

torch.Size([32, 4])

torch.Size([32, 4])

torch.Size([32, 4])

torch.Size([32, 4])

torch.Size([32, 4])

torch.Size([32, 4])

torch.Size([32, 4])
tensor(1.9872e-05, grad_fn=<MulBackward0>)


In [43]:
## Measure distortion in Ws
tensor_magnitudes = [torch.abs(tensor) for tensor in Ws]
stacked_tensors = torch.stack(tensor_magnitudes)
std_dev_across_tensors = torch.std(stacked_tensors, dim=0)
torch.norm(std_dev_across_tensors)

tensor(0.0020, grad_fn=<LinalgVectorNormBackward0>)